# Customer Segmentation & Marketing Response Prediction
## Comprehensive End-to-End Machine Learning Analysis

This notebook provides a complete data story covering:
1. **Exploratory Data Analysis (EDA) & Data Cleaning** (Missing value imputation, outlier handling, categorical normalization)
2. **Feature Engineering** (Aggregated spending, purchase channels, family structure, tenure, and engagement ratios)
3. **Model 1: Unsupervised Customer Segmentation** (StandardScaler, Elbow Method, Silhouette Optimization, K-Means, and 2D PCA Projection)
4. **Dynamic Cluster Profiling** (Centroid analysis and persona naming)
5. **Model 2: Supervised Campaign Response Prediction** (Zero data leakage architecture, Logistic Regression, Random Forest, XGBoost)
6. **Dual-Model Integration & Personalized Marketing Recommendations**

In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import load_raw_data, clean_data
from src.feature_engineering import engineer_features
from src.clustering import evaluate_optimal_k, fit_clustering_pipeline, CLUSTERING_FEATURES
from src.prediction import train_and_evaluate_models
from src.recommendations import generate_recommendation

sns.set_theme(style="whitegrid")
print("Environment initialized.")

### Step 1: Load and Clean Dataset
We inspect missing values in `Income`, remove extreme outliers, drop constant columns (`Z_CostContact`, `Z_Revenue`), and convert `Dt_Customer` into a proper datetime timestamp.

In [ ]:
raw_df = load_raw_data("../data/customers.csv")
clean_df, clean_report = clean_data(raw_df)
print("Cleaning Report:", clean_report)
clean_df.head(3)

### Step 2: Feature Engineering
We compute domain-specific features:
- `Total_Spending`: aggregate spend across all six categories
- `Total_Purchases`: total transactions across web, catalog, store, and deals
- `Customer_Age` and `Customer_Tenure`
- Purchase channel ratios and spending shares

In [ ]:
df_feat = engineer_features(clean_df)
print("Features created:", [c for c in df_feat.columns if c not in clean_df.columns])
df_feat[['Total_Spending', 'Total_Purchases', 'Customer_Age', 'Customer_Tenure', 'Web_Purchase_Ratio']].describe()

### Step 3: Model 1 — Customer Segmentation (K-Means)
Evaluate K from 2 to 8 using the Elbow Method and Silhouette Score to discover the optimal cluster count.

In [ ]:
k_eval = evaluate_optimal_k(df_feat, k_range=(2, 8))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(k_eval["k_values"], k_eval["inertias"], marker="o", color="#2563eb")
ax1.set_title("Elbow Method (Inertia)")
ax1.set_xlabel("K")
ax1.set_ylabel("Inertia")

ax2.plot(k_eval["k_values"], k_eval["silhouette_scores"], marker="o", color="#16a34a")
ax2.set_title("Silhouette Score vs K")
ax2.set_xlabel("K")
ax2.set_ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

### Step 4: Fit K-Means Pipeline & 2D PCA Visualization
We fit K-Means with $K=4$ clusters, apply PCA to project into 2D space, and dynamically profile each cluster.

In [ ]:
kmeans, scaler, pca, segmented_df, profiles = fit_clustering_pipeline(df_feat, n_clusters=4)

for c_id, prof in profiles.items():
    print(f"Cluster {c_id} ({prof['name']}): {prof['count']} customers, Avg Spend: ${prof['spending']}, Avg Income: ${prof['income']}")

plt.figure(figsize=(10, 6))
sns.scatterplot(data=segmented_df, x="PCA1", y="PCA2", hue="Cluster_Name", palette="tab10", alpha=0.8)
plt.title("Customer Segments Projected on 2D PCA Space")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

### Step 5: Model 2 — Supervised Campaign Response Prediction
We train Logistic Regression, Random Forest, and XGBoost on unseen test holdouts (80/20 stratified split) while strictly preventing data leakage.

In [ ]:
pred_results = train_and_evaluate_models(segmented_df, include_cluster=True)
print("Model Comparison Table:")
display(pred_results["comparison_table"])
print("\nRecommended Model:", pred_results["best_model_name"])

### Step 6: Dual-Model Integration & Personalized Action
Demonstrating end-to-end inference for a target customer.

In [ ]:
sample_customer = segmented_df.iloc[0].to_dict()
best_model = pred_results["models"][pred_results["best_model_name"]]["model"]
preprocessor = pred_results["preprocessor"]

resp_pred = predict_customer_response(
    sample_customer, best_model, preprocessor,
    pred_results["feature_names"]["all"]
)

rec = generate_recommendation(
    cluster_name=sample_customer["Cluster_Name"],
    response_probability=resp_pred["probability"],
    is_responsive=resp_pred["prediction"] == 1,
    customer_data=sample_customer
)

print("Customer ID:", sample_customer.get("Id", "N/A"))
print("Assigned Segment:", rec["segment"])
print("Response Probability:", f"{rec['response_probability_pct']}%")
print("Outcome:", rec["predicted_label"])
print("Recommended Strategy:", rec["primary_strategy"])
print("Offer:", rec["campaign_offer"])
print("Preferred Channel:", rec["recommended_channel"])